In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_1samp, wilcoxon

DATA_PATH = "results/Shoes - Image Assessment.csv"

# Names shown for the two anonymized options in the survey form.
OPTION_1 = "JuggXL LoRA"
OPTION_2 = "Professional Photos"
COLORS = ["#4a90e2", "#7f8c8d"]

In [ ]:
df = pd.read_csv(DATA_PATH)

# The 13 comparison questions share one title; pandas suffixes repeats as .1, .2, ...
response_cols = [column for column in df.columns if column.startswith("Which image looks better?")]
assert len(response_cols) == 13, f"Expected 13 comparison questions, found {len(response_cols)}"
assert df[response_cols].isin(["Option 1", "Option 2"]).all().all(), "Unexpected response values"

print(f"Participants: {len(df)}")

In [ ]:
# Participant-level tests: each participant's share of choices favouring Option 2.
n_questions = len(response_cols)
option2_choices = (df[response_cols] == "Option 2").sum(axis=1)
pref_option2 = option2_choices / n_questions
print(f"Mean preference for {OPTION_2}: {pref_option2.mean():.3f}")

t_res = ttest_1samp(pref_option2, 0.5)
print(f"One-sample t-test vs 0.5: t = {t_res.statistic:.3f}, p = {t_res.pvalue:.3g}")

# Rank integer differences (2k - n, proportional to k/n - 0.5) so participants equally far
# from 50% tie exactly; float proportions such as 6/13 and 7/13 round differently and
# would silently break those ties, changing W and p.
w_res = wilcoxon(2 * option2_choices - n_questions)
print(f"Wilcoxon signed-rank test vs 0.5: W = {w_res.statistic:.1f}, p = {w_res.pvalue:.3g}")

In [ ]:
count_option1 = int((df[response_cols] == "Option 1").sum().sum())
count_option2 = int((df[response_cols] == "Option 2").sum().sum())

print(f"{OPTION_1}: {count_option1} votes")
print(f"{OPTION_2}: {count_option2} votes")
print(f"Total: {count_option1 + count_option2} votes")

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))
plt.pie(
    [count_option1, count_option2],
    labels=[OPTION_1, OPTION_2],
    autopct="%1.1f%%",
    colors=COLORS,
)
plt.title(
    "Fashion Retail Case Study: Leading Web-Scale GenAI vs. Professional Photography",
    fontsize=14,
)
plt.tight_layout()
plt.savefig("results/SOTAvsPhotographer.png", format="png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
sources = [OPTION_1, OPTION_2]
counts = [count_option1, count_option2]

plt.figure(figsize=(8, 6))
sns.barplot(x=sources, y=counts, hue=sources, palette=COLORS, legend=False)
plt.title(f"Preference Counts for {OPTION_1} vs {OPTION_2}", fontsize=16)
plt.xlabel("Image Source", fontsize=14)
plt.ylabel("Number of Votes", fontsize=14)

for index, count in enumerate(counts):
    plt.text(index, count * 1.02, f"{count:,}", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()